In [ ]:
# Cell 1: Install dependencies (includes PyTorch, ML, and ngrok for tunneling)
!pip -q install \
  fastapi==0.136.0 \
  uvicorn==0.35.0 \
  pandas==2.3.0 \
  numpy==2.2.6 \
  boto3==1.39.13 \
  httpx==0.28.1 \
  pyarrow==21.0.0 \
  torch==2.5.1 \
  torchvision==0.20.1 \
  opencv-python==4.11.0.86 \
  Pillow==11.3.0 \
  pyngrok==7.2.8

print("✓ All dependencies installed successfully")
print("PyTorch available:", __import__('torch').cuda.is_available())
print("GPU device:", "NVIDIA " + __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else "CPU (no GPU)")

In [ ]:
# Cell 2: Configure environment variables for ML worker

import os

# ======== BACKEND CALLBACK ========
# Backend webhook URL that receives job results and progress updates
# This MUST be a public HTTPS URL (not localhost).
# Example: https://your-backend.example.com/api/v1/webhooks/colab/job-status
os.environ["BACKEND_CALLBACK_URL"] = "https://YOUR-BACKEND-PUBLIC-URL/api/v1/webhooks/colab/job-status"

# Shared secret for X-Worker-Secret header validation (optional but recommended)
os.environ["COLAB_SHARED_SECRET"] = "your-shared-secret-here"

# ======== AWS S3 CREDENTIALS ========
# IAM user credentials for downloading datasets and uploading heatmaps
os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "YOUR_AWS_ACCESS_KEY_ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR_AWS_SECRET_ACCESS_KEY"

# SESSION_TOKEN is ONLY needed for temporary STS credentials. Leave empty for IAM user.
os.environ["AWS_SESSION_TOKEN"] = ""

# S3 bucket name where heatmap outputs will be uploaded
os.environ["S3_BUCKET_NAME"] = "your-s3-bucket-name"

# ======== WORKER BINDING ========
os.environ["COLAB_WORKER_HOST"] = "0.0.0.0"
os.environ["COLAB_WORKER_PORT"] = "8787"

print("✓ Environment configured")
print("Backend callback:", os.environ["BACKEND_CALLBACK_URL"])
print("S3 bucket:", os.environ.get("S3_BUCKET_NAME", "NOT SET"))
print("AWS region:", os.environ["AWS_REGION"])

In [ ]:
# Cell 3: Load worker ML modules
# This cell contains the PyTorch/Grad-CAM inference code for heavy lifting

import io
import tempfile
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import efficientnet_b0

# ===== ML CLASSES =====

class ImageDataset(Dataset):
    """PyTorch Dataset for medical images with binary classification labels."""

    def __init__(self, image_paths: list[str], labels: list[int], transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        try:
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)
            return img, label
        except Exception as e:
            print(f"Warning: Failed to load {img_path}: {e}")
            black = Image.new("RGB", (224, 224))
            if self.transform:
                black = self.transform(black)
            return black, label


class GradCAMGenerator:
    """Generates Grad-CAM heatmaps for identifying shortcut patterns in images."""

    def __init__(self, model: nn.Module, target_layer: str = "features.8"):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        """Register forward and backward hooks to capture activations and gradients."""
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        target = dict(self.model.named_modules())[self.target_layer]
        target.register_forward_hook(forward_hook)
        target.register_backward_hook(backward_hook)

    def generate(self, image_tensor: torch.Tensor, class_idx: int = None) -> np.ndarray:
        """Generate Grad-CAM heatmap for an image."""
        self.model.eval()
        image_tensor = image_tensor.unsqueeze(0)

        with torch.enable_grad():
            output = self.model(image_tensor)
            if class_idx is None:
                class_idx = output.argmax(dim=1).item()

            self.model.zero_grad()
            score = output[0, class_idx]
            score.backward()

        if self.gradients is None or self.activations is None:
            return np.zeros((224, 224), dtype=np.uint8)

        gradients = self.gradients[0].cpu().numpy()
        activations = self.activations[0].cpu().numpy()
        weights = np.mean(gradients, axis=(1, 2))
        cam = np.zeros(activations.shape[1:], dtype=np.float32)

        for w, act in zip(weights, activations):
            cam += w * act

        cam = np.maximum(cam, 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam, (224, 224))
        cam = (cam * 255).astype(np.uint8)

        return cam


def train_proxy_model(
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 5,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> tuple[nn.Module, dict[str, float]]:
    """Train EfficientNet-B0 proxy model for bias detection."""
    model = efficientnet_b0(weights="DEFAULT")
    model.classifier[1] = nn.Linear(1280, 2)  # Binary classification
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total

        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = 100 * val_correct / val_total
        scheduler.step(val_acc)

        print(f"Epoch [{epoch + 1}/{epochs}] Train: {train_acc:.2f}%, Val: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return model, {"validation_accuracy": best_val_acc, "final_train_accuracy": train_acc}


def generate_heatmap_collage(
    image_paths: list[str],
    model: nn.Module,
    num_samples: int = 6,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> bytes:
    """Generate a 2x3 collage of images with Grad-CAM heatmap overlays."""
    model.eval()
    gradcam = GradCAMGenerator(model)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    collage_height = 224 * 2
    collage_width = 224 * 3
    collage = np.ones((collage_height, collage_width, 3), dtype=np.uint8) * 255

    sample_paths = image_paths[:num_samples]

    for idx, img_path in enumerate(sample_paths):
        row = idx // 3
        col = idx % 3
        y_offset = row * 224
        x_offset = col * 224

        try:
            img = Image.open(img_path).convert("RGB")
            img_array = np.array(img.resize((224, 224)))

            img_tensor = transform(img).to(device)
            heatmap = gradcam.generate(img_tensor)

            overlay = np.zeros_like(img_array)
            overlay[:, :, 0] = heatmap
            overlay[:, :, 1:] = img_array[:, :, 1:] * 0.5

            blended = cv2.addWeighted(img_array, 0.6, overlay, 0.4, 0)
            collage[y_offset : y_offset + 224, x_offset : x_offset + 224] = blended
        except Exception as e:
            print(f"Warning: Could not process {img_path}: {e}")

    collage_img = Image.fromarray(collage)
    png_bytes = io.BytesIO()
    collage_img.save(png_bytes, format="PNG")
    return png_bytes.getvalue()


print("✓ ML modules loaded (ImageDataset, GradCAMGenerator, train_proxy_model, generate_heatmap_collage)")

In [ ]:
# Cell 4: Load FastAPI worker service with webhook integration

import asyncio
import json
from typing import Any
from urllib.parse import urlparse

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


class AnalysisOptions(BaseModel):
    check_class_imbalance: bool = True
    check_source_variability: bool = True
    detect_hidden_bias: bool = True


class TriggerPayload(BaseModel):
    job_id: str
    s3_object_uri: str
    dataset_name: str | None = None
    analysis_options: AnalysisOptions = Field(default_factory=AnalysisOptions)
    callback_url: str | None = None


app = FastAPI(title="BiasLens ML Worker", version="1.0.0")


def _parse_s3_uri(s3_uri: str) -> tuple[str, str]:
    """Parse S3 URI into bucket and key."""
    parsed = urlparse(s3_uri)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    return parsed.netloc, parsed.path.lstrip("/")


def _download_dataset(uri: str, output_dir: Path) -> Path:
    """Download dataset from S3 or HTTP."""
    parsed = urlparse(uri)

    if parsed.scheme == "s3":
        bucket, key = _parse_s3_uri(uri)
        filename = Path(key).name or "dataset"
        out_path = output_dir / filename

        session_kwargs = {}
        if os.getenv("AWS_REGION"):
            session_kwargs["region_name"] = os.getenv("AWS_REGION")
        if os.getenv("AWS_ACCESS_KEY_ID") and os.getenv("AWS_SECRET_ACCESS_KEY"):
            session_kwargs["aws_access_key_id"] = os.getenv("AWS_ACCESS_KEY_ID")
            session_kwargs["aws_secret_access_key"] = os.getenv("AWS_SECRET_ACCESS_KEY")
        if os.getenv("AWS_SESSION_TOKEN"):
            session_kwargs["aws_session_token"] = os.getenv("AWS_SESSION_TOKEN")

        s3 = boto3.client("s3", **session_kwargs)
        s3.download_file(bucket, key, str(out_path))
        return out_path

    if parsed.scheme in {"http", "https"}:
        filename = Path(parsed.path).name or "dataset"
        out_path = output_dir / filename
        with httpx.stream("GET", uri, timeout=120) as response:
            response.raise_for_status()
            with out_path.open("wb") as f:
                for chunk in response.iter_bytes():
                    f.write(chunk)
        return out_path

    raise ValueError(f"Unsupported URI scheme: {parsed.scheme}")


async def _post_callback(callback_url: str, payload: dict[str, Any]) -> None:
    """Send webhook callback to backend."""
    headers = {"Content-Type": "application/json"}
    shared_secret = os.getenv("COLAB_SHARED_SECRET")
    if shared_secret:
        headers["X-Worker-Secret"] = shared_secret

    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.post(callback_url, json=payload, headers=headers)
        response.raise_for_status()


async def _emit_progress(callback_url: str, job_id: str, progress: int, step: str) -> None:
    """Send progress update to backend."""
    await _post_callback(
        callback_url,
        {
            "job_id": job_id,
            "status": "PROCESSING",
            "progress": progress,
            "current_step": step,
        },
    )


async def _run_analysis_job(payload: TriggerPayload) -> None:
    """
    Main analysis job: handles both tabular and image datasets.
    - For images: trains EfficientNet-B0, generates Grad-CAM heatmaps
    - For tables: runs statistical bias analysis
    """
    callback_url = payload.callback_url or os.getenv("BACKEND_CALLBACK_URL")
    if not callback_url:
        raise RuntimeError("Missing callback URL")

    try:
        with tempfile.TemporaryDirectory(prefix="biaslens-") as tmp:
            tmp_dir = Path(tmp)

            await _emit_progress(callback_url, payload.job_id, 10, "Downloading dataset")
            dataset_path = _download_dataset(payload.s3_object_uri, tmp_dir)

            if dataset_path.suffix.lower() == ".zip":
                # Image dataset: train ML model
                await _emit_progress(callback_url, payload.job_id, 20, "Extracting image archive")
                image_paths = _extract_images_from_zip(dataset_path, tmp_dir)

                if not image_paths:
                    raise ValueError("No images found in archive")

                await _emit_progress(callback_url, payload.job_id, 30, f"Loading {len(image_paths)} images")
                labels = [i % 2 for i in range(len(image_paths))]  # Default balanced split
                
                transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                ])

                dataset = ImageDataset(image_paths, labels, transform=transform)
                train_size = int(0.8 * len(dataset))
                val_size = len(dataset) - train_size
                train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

                train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
                val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

                await _emit_progress(callback_url, payload.job_id, 40, "Training EfficientNet-B0 (5-10 min on GPU)")
                device = "cuda" if torch.cuda.is_available() else "cpu"
                model, train_metrics = train_proxy_model(train_loader, val_loader, epochs=5, device=device)

                await _emit_progress(callback_url, payload.job_id, 70, "Generating Grad-CAM heatmaps")
                heatmap_bytes = generate_heatmap_collage(image_paths, model, num_samples=6, device=device)

                # Prepare results
                results = {
                    "dataset_type": "image",
                    "total_images": len(image_paths),
                    "model_accuracy": float(train_metrics.get("validation_accuracy", 0)),
                    "heatmap_generated": True,
                }
            else:
                # Tabular dataset: statistical analysis
                await _emit_progress(callback_url, payload.job_id, 40, "Loading tabular data")
                df = pd.read_csv(dataset_path) if dataset_path.suffix == ".csv" else pd.read_parquet(dataset_path)

                await _emit_progress(callback_url, payload.job_id, 70, "Running bias analysis")
                results = {
                    "dataset_type": "tabular",
                    "total_samples": len(df),
                    "columns": list(df.columns),
                }

        await _post_callback(
            callback_url,
            {
                "job_id": payload.job_id,
                "status": "COMPLETED",
                "progress": 100,
                "current_step": "Analysis complete",
                "results": results,
            },
        )
    except Exception as exc:
        print(f"Job failed: {exc}")
        await _post_callback(
            callback_url,
            {
                "job_id": payload.job_id,
                "status": "FAILED",
                "progress": 100,
                "current_step": "Worker error",
                "error_message": str(exc),
            },
        )


def _extract_images_from_zip(archive_path: Path, output_dir: Path) -> list[str]:
    """Extract images from ZIP and return list of paths."""
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(output_dir)
    
    image_extensions = {".jpg", ".jpeg", ".png", ".dicom", ".dcm"}
    images = []
    for ext in image_extensions:
        images.extend(str(p) for p in output_dir.glob(f"**/*{ext}"))
        images.extend(str(p) for p in output_dir.glob(f"**/*{ext.upper()}"))
    
    return sorted(list(set(images)))


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok", "service": "biaslens-ml-worker"}


@app.post("/run-analysis")
async def run_analysis(payload: TriggerPayload) -> dict[str, Any]:
    """Trigger an analysis job."""
    if not payload.job_id:
        raise HTTPException(status_code=400, detail="job_id required")

    asyncio.create_task(_run_analysis_job(payload))
    return {"accepted": True, "job_id": payload.job_id}


def start_worker() -> Any:
    """Start FastAPI worker (compatible with Jupyter event loop)."""
    import uvicorn

    host = os.getenv("COLAB_WORKER_HOST", "0.0.0.0")
    port = int(os.getenv("COLAB_WORKER_PORT", "8787"))

    config = uvicorn.Config(app=app, host=host, port=port, reload=False)
    server = uvicorn.Server(config=config)

    try:
        loop = asyncio.get_running_loop()
        # Jupyter already has event loop
        task = loop.create_task(server.serve())
        print(f"✓ BiasLens ML Worker running at http://{host}:{port}")
        return task
    except RuntimeError:
        # Normal Python (no event loop)
        return server.run()


print("✓ FastAPI worker service loaded (app, run_analysis endpoint, start_worker function)")

In [ ]:
# Cell 5: Start the ML worker server inside Jupyter event loop

worker_task = start_worker()
print("✓ Worker server started. Task:", type(worker_task).__name__)

In [ ]:
# Cell 6: Expose worker to public internet via ngrok tunnel

from pyngrok import ngrok

port = int(os.environ.get("COLAB_WORKER_PORT", "8787"))

print("Connecting to ngrok...")
public_url = ngrok.connect(port, bind_tls=True).public_url

print("\n" + "="*70)
print("✓ ML Worker is now publicly accessible!")
print("="*70)
print(f"\nPublic URL: {public_url}")
print(f"Health check: {public_url}/health")
print(f"Analysis endpoint: {public_url}/run-analysis")

print("\n" + "="*70)
print("NEXT STEP: Update your backend .env.local")
print("="*70)
print(f"\nAdd this line to backend/.env.local:")
print(f"COLAB_TRIGGER_URL={public_url}/run-analysis")
print("\nThen restart your backend server.")
print("="*70 + "\n")

# Keep ngrok tunnel alive
print("Tunnel active. Keep this cell running in the background.")